### Coleta de dados de Temperatura, Umidade e Precipitação

Será utilizado o dataset derived-era5-single-levels-daily-statistics do ERA5

Documentação em:
https://cds.climate.copernicus.eu/datasets/derived-era5-single-levels-daily-statistics?tab=documentation

Os dados extraídos:
<pre>
- Umidade   -> variável 2m_dewpoint_temperature (ponto de orvalho),
               O processo de conversão para percentual será detalhado no passo que executa a conversão
</pre>

Os dados serão coletados por Ano e Mês 

Os dados requisitados estão no retangulo geográfico geográfico [6, -74, -34, -35] -> [Norte, Oeste, Sul, Leste] em graus onde está o Brasil


In [1]:
import cdsapi
import os, sys
import xarray as xr
from pyspark.sql import functions as F

In [ ]:
# Cria a conexão Spark

# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
from spark_utils import get_spark_session # ver em C:\Marco Conti\Projetos\MAIS-v2\spark_utils.py
spark = get_spark_session("MeuNotebook")

In [ ]:
# Os arquivos utilizados durante o processamento serão removidos no final do notebook
remover_arquivos = []

Requisição dos dados da variável derived-era5-single-levels-daily-statistics do ERA5 utilizando a biblioteca cdsapi

In [2]:
PROJECT_PATH   = os.getcwd()
DATA_PATH_ROOT = "C:\\Marco Conti\\Projetos\\Dados\\"

print(PROJECT_PATH)

c:\Marco Conti\Projetos\mais_einstein\Ondas_Calor


In [3]:
import datetime

for year in range(2000, 2001): # Ajuste o intervalo de anos conforme necessário
    start = datetime.datetime.now()
    print("Start download - year : ",year, " - ", start, end="" )    

    dataset = "derived-era5-single-levels-daily-statistics"
    request = {
        "product_type": "reanalysis",
        "variable": [
            "2m_dewpoint_temperature"
        ],
        "year": f"{year}",
        "month": ["01","02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12"],
        "day": ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12", "13", "14", "15"
            ,"16", "17", "18", "19", "20", "21", "22", "23", "24", "25", "26", "27", "28", "29", "30", "31"
        ],
        "daily_statistic": "daily_mean",
        "time_zone": "utc-03:00",
        "frequency": "1_hourly",
        # Retangulo geográfico definido por Norte, Oeste, Sul e Leste em graus onde está o Brasil
        "area": [6      # Norte
                ,-74    # Oeste
                ,-34    # Sul
                ,-38]   # Leste
    }

    # Informações de autenticação estão em:
    # C:\Users\DRT90628\.ecmwfdatastoresrc
    # *** Criar um novo contrato de autenticação deverá ser criado usando um usuário de serviços do Einstein

    client = cdsapi.Client(
        url = os.getenv("ECMWF_DATASTORES_URL"),
        key = os.getenv("ECMWF_DATASTORES_KEY"),
    )

    ret_download = client.retrieve(dataset, request).download()
    remover_arquivos.append(ret_download)

    os.rename(ret_download, f"{DATA_PATH_ROOT}\ERA5-Umidade\ERA5_umidade_{year}.nc")

    final = datetime.datetime.now()
    print(" - End download - year : ",year, " - ", final, " - Duration: ", final-start, "\n")


Start download - year :  2000  -  2026-08-03 13:24:22.052987

2026-08-03 13:24:25,043 INFO Request ID is 367e6521-75b2-41ad-aee3-a7f383f42371
2026-08-03 13:24:25,242 INFO status has been updated to accepted
2026-08-03 18:11:41,591 INFO status has been updated to successful


NameError: name 'remover_arquivos' is not defined

Esta função ira converter os dados dos arquivos .nc para o format Dask para então converter para Dataframe Spark <br>
Isso deixa o processamento em paralelo e será importante para processamento de grandes volumes (1 ano com todos os meses e dias)

In [ ]:
def convert_netcdf4_Spark(file_name):
    with xr.open_dataset(f"C:\\Marco Conti\\Projetos\\MAIS-v2\\Ondas_Calor\\{file_name}"
                        ,engine="netcdf4"
                        ,chunks={"time": 365
                                ,"latitude": 100
                                ,"longitude": 100 }
                        ) as ds:
        
        # Transforma o Dataset em um Spark Dataframe
        df_dask   = ds.to_dask_dataframe()
        df_dask_c = df_dask.compute()
        df_spark  = spark.createDataFrame(df_dask_c)    
    return df_spark

In [ ]:
df_ponto_orvalho = convert_netcdf4_Spark(ret_download)

In [ ]:
drop_cols = ["valid_time", "number", "d2m"]
df_ponto_orvalho = \
    (df_ponto_orvalho
        .withColumns({"indicador": F.lit("ponto_orvalho")
                            ,"valor": (F.col("d2m") - F.lit(273.15)).cast('double')
                            ,"unidade_medida": F.lit("celsius")
                            ,"data_medicao": F.col("valid_time").cast("date")}
                            )
            .drop(*drop_cols)
    )

df_ponto_orvalho.printSchema()
df_ponto_orvalho.show()

Faz a junção dos dados de temperatra e ponto de orvalho para calcular o percentual da umidade:

In [ ]:
df_temperatura = spark.read.parquet(r"C:\Marco Conti\Projetos\MAIS-v2\dados\ERA5-temperaturas\ERA5_temperatura.parquet")
print("Temperatura: ", df_temperatura.count())          # 783.587
print("Ponto de orvalho:", df_ponto_orvalho.count())    # 379.155

In [ ]:
df_temp_ponto_orvalho = \
    (df_temperatura.alias('t')
        .join(df_ponto_orvalho.alias('p')
             ,((F.col('t.data_medicao') == F.col('p.data_medicao')) & 
               (F.col("t.latitude")     == F.col("p.latitude")) & 
               (F.col("t.longitude")    == F.col("p.longitude")))
             ,'inner')
        .select('t.data_medicao'
               ,'t.latitude'
               ,'t.longitude'
               ,F.col('t.valor').alias('temperatura_celsius')
               ,F.col('p.valor').alias('temp_ponto_orvalho_celsius'))
    )


print("Join:", df_temp_ponto_orvalho.count())

In [ ]:
df_temp_ponto_orvalho.limit(10).show(truncate=False)

Faz a conversão de temperatura para percentual de umidade usando a equação de Magnus-Tetens.

https://en.wikipedia.org/wiki/Tetens_equation


In [ ]:

# Constantes da equação de Magnus-Tetens.
A = 17.67
B = 243.5

df_umidade_Magnus_Tetens = (
    df_temp_ponto_orvalho
        .withColumn("indicador", F.lit("umidade"))
        .withColumn("unidade_medida", F.lit("percentual"))
        .withColumn("valor",
            F.round(  
                F.lit(100.0) * F.exp(
                    (
                        A * F.col("temp_ponto_orvalho_celsius") /
                        (F.col("temp_ponto_orvalho_celsius") + B)
                    )
                    -
                    (
                        A * F.col("temperatura_celsius") /
                        (F.col("temperatura_celsius") + B)
                    )
                )
            ,4)
        )
).drop("temperatura_celsius", "temp_ponto_orvalho_celsius")

df_umidade_Magnus_Tetens.show()

In [ ]:
df_umidade = \
    (df_umidade_Magnus_Tetens
        .select("data_medicao"
               ,"latitude"
               ,"longitude"
               ,"indicador"
               ,"valor"
               ,"unidade_medida"))

In [ ]:
# df_umidade.toPandas().to_csv("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\ERA5-temperaturas\\ERA5_umidade.csv", index=False)

df_umidade.toPandas().to_parquet("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\ERA5-temperaturas\\ERA5_umidade.parquet")

In [ ]:
# df_csv = spark.read.csv("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\ERA5-temperaturas\\ERA5_umidade_temperatura_precipitacao.csv", header=True, inferSchema=True)
# print("df_csv:", df_csv.count())
# df_csv.printSchema()
# df_csv.show(10,False)


df_parquet = spark.read.parquet("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\ERA5-temperaturas\\ERA5_umidade.parquet")
print("df_csv:", df_parquet.count())
df_parquet.printSchema()
df_parquet.show(10,False)


In [ ]:
# **** INCLUIR EXCLUSÃO DE ARQUIVOS (.zip e .nc)

for file in remover_arquivos:
    print("Arquivo:", file, end="")
    os.remove(file)
    print(" Removido com sucesso")
